# H8 HazardTwin · Galați · CP3: rezultate v1 + scenariul de alertă pe Galați

Rulează **codul exact din repo** [IchimCosmin/h8-hazardtwin-galati](https://github.com/IchimCosmin/h8-hazardtwin-galati) (clonat mai jos; commit-ul e afișat):

1. `ingest` + `learn`: SEN12FLOOD, metoda vs. baseline-uri pe 5 split-uri spațiale (`results_v1_H8.csv`, identic cu v1);
2. `galati`: seria Sentinel-1 peste județul Galați din Microsoft Planetary Computer (`sentinel-1-rtc`, fără cont), M2 antrenat pe tot SEN12FLOOD, probabilitate de inundație pe 80 de tile-uri × fiecare trecere, scenariul de alertă verificat pe inundațiile din 13–14 sept. 2024 (Copernicus EMS EMSR758).

Necesită dataset-ul Kaggle *SEN12FLOOD: Flood Detection Dataset* și **Internet on**.

In [ ]:
import glob, os, subprocess, sys, time
REPO = '/kaggle/working/h8-hazardtwin-galati'
OUT = '/kaggle/working/out'
os.makedirs(OUT, exist_ok=True)
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/IchimCosmin/h8-hazardtwin-galati', REPO], check=True)
print('commit:', subprocess.run(['git', '-C', REPO, 'log', '-1', '--format=%h %ad %s', '--date=short'], capture_output=True, text=True).stdout)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pystac-client', 'planetary-computer', 'bottleneck'], check=True)
sys.path.insert(0, REPO + '/src')
from h8twin.cli import main
ROOT = os.path.dirname(sorted(glob.glob('/kaggle/input/**/S1list.json', recursive=True), key=len)[0])
print('SEN12FLOOD root:', ROOT, '| CPUs:', os.cpu_count())

## 1. SEN12FLOOD: ingestie și învățare (rezultate v1)

In [ ]:
t0 = time.time()
main(['ingest', '--root', ROOT, '--out', OUT, '--n-pixels', '60'])
main(['learn', '--root', ROOT, '--out', OUT, '--seeds', '5', '--jobs', '-1'])
import hashlib
s = open(f'{OUT}/results_v1_H8.csv').read()
print('results_v1_H8.csv md5', hashlib.md5(s.encode()).hexdigest(), f'| {time.time()-t0:.0f}s')

## 2. Galați: seria Sentinel-1 și scenariul de alertă

In [ ]:
t0 = time.time()
main(['galati', '--out', OUT, '--start', '2023-09-01', '--end', '2025-03-01'])
print(f'galati: {time.time()-t0:.0f}s')

In [ ]:
import pandas as pd
pd.set_option('display.width', 250)
per = pd.read_csv(f'{OUT}/galati_passes.csv', parse_dates=['date'])
print(per[per.date.between('2024-08-01', '2024-11-15')].to_string(index=False))
sc = pd.read_csv(f'{OUT}/galati_tile_scores.csv', parse_dates=['date'])
tiles = pd.read_csv(f'{OUT}/galati_tiles.csv')
ev = sc[sc.date.between('2024-09-13', '2024-09-30') & sc.scored].merge(tiles[['tile', 'lon', 'lat', 'place']], on='tile')
print(ev.sort_values('p_flood', ascending=False).head(15).round(3).to_string(index=False))